In [2]:
import json
import os
import random
import re
from time import sleep

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [ ]:
HEADERS = {
    "User-Agent": "LitaratureRecommender/1.0 (contact: tomasz.bialecki@student.put.poznan.pl)"
}

TARGET_DIR = "data/scraping/"
OUTPUT_FILE = TARGET_DIR + "scraped_data.jsonl"
ADDITIONAL_LINKS_FILE = TARGET_DIR + "additional_links.txt"
TARGET_ARTICLE_COUNT = 1200
REQUEST_DELAY_MULTIPLIER = 3

In [4]:
STARTING_URLS = {
    "Le Monde" : "https://en.wikipedia.org/wiki/Le_Monde%27s_100_Books_of_the_Century",
    "Nobel Prize" : "https://en.wikipedia.org/wiki/List_of_Nobel_laureates_in_Literature",
    "Pulitzer Prize" : "https://en.wikipedia.org/wiki/Pulitzer_Prize_for_Fiction"
}

In [5]:
links_to_books_lm = []
links_to_authors_lm = []
links_to_authors_n = []
links_to_books_p = []
links_to_authors_p = []

In [ ]:
def extract_links(rows, lists):

    for row in rows:
        cols = row.find_all("td")
        if not cols:
            continue
        try:

            if len([cols[col_index].find("a").get("href") for col_index, _ in lists]) == len(lists):
                for col_index, link_list in lists:
                    link = cols[col_index].find("a").get("href")
                    link_list.append(link)
            
        except AttributeError:
            continue

In [7]:
def is_valid_book_link(href: str | None) -> bool:
    """Filters out footnotes, system pages, and publication year links."""
    if not href or not href.startswith("https://en.wikipedia.org/wiki/"):
        return False
    if "#" in href:
        return False

    disallowed_prefixes = (
        "/wiki/Wikipedia:",
        "/wiki/Category:",
        "/wiki/Special:",
        "/wiki/Help:",
        "/wiki/File:",
        "/wiki/Talk:",
        "/wiki/Portal:",
        "/wiki/Template:",
    )
    if any(href.startswith("https://en.wikipedia.org" + p) for p in disallowed_prefixes):
        return False

    return not re.search(r"/(1[789]\d\d|20\d\d)(_in_literature)?$", href)

In [8]:
def extract_book_links_from_author(author_link):
    works_links = set()

    response = requests.get(author_link, headers=HEADERS)
    parsed = BeautifulSoup(response.text, "html.parser")

    infobox = parsed.find("table", class_="infobox")
    if infobox:
        for tr in infobox.find_all("tr"):
            label = tr.find(["th", "td"], class_="infobox-label")
            clean_row_text = re.sub(
                    r"\s+", " ", tr.get_text()
                ).strip().lower()
            if label and "notable work" in clean_row_text:
                data = tr.find("td", class_="infobox-data")
                if data:
                    for a in data.find_all("a"):
                        if is_valid_book_link(a.get("href")):
                            works_links.add(a.get("href"))
    return works_links

In [9]:
def expand_list_of_links(links_to_scrape, links_to_authors):
    links_to_scrape_set = set(links_to_scrape)
    with open(ADDITIONAL_LINKS_FILE, "a", encoding="utf-8") as f:
        for i, author_link in enumerate(links_to_authors):

            if i % 5 == 0:
                print(f"Processing author [{i+1}/{len(links_to_authors)}]: {author_link}")

            works_links = extract_book_links_from_author(author_link)
            if works_links:
                links_to_scrape_set.update(works_links)

                for link in works_links:
                    f.write(link + "\n")
                    f.flush()
            sleep(random.random() * REQUEST_DELAY_MULTIPLIER)

    return list(links_to_scrape_set)

In [10]:
def get_links_to_scrape():
    for source_name, url in STARTING_URLS.items():
        sleep(0.3)
        response = requests.get(url, headers=HEADERS)
        parsed = BeautifulSoup(response.text, "html.parser")

        match source_name:
            case "Le Monde":
                rows = parsed.find("tbody").find_all("tr")
                extract_links(rows, [(1, links_to_books_lm), (2, links_to_authors_lm)])
                
            case "Nobel Prize":
                rows = parsed.find("section", {"aria-labelledby": "Laureates"}).find("tbody").find_all("tr")
                extract_links(rows, [(2, links_to_authors_n)])

            case "Pulitzer Prize":
                rows = parsed.find("section", {"aria-labelledby": "Winners"}).find('table', {"class": "wikitable"}).find_all("tr")
                extract_links(rows, [(0, links_to_authors_p), (1, links_to_books_p)])

    return list(set(links_to_books_lm + links_to_authors_lm + links_to_authors_n + links_to_books_p + links_to_authors_p))

In [11]:
def clean_content(content):
    unwanted_selectors = [
        "sup.reference",  # Citation numbers like [1], [2]
        "span.mw-editsection",  # [edit] links
        "div.reflist",  # Reference blocks at the bottom
        "div.navbox",  # Footer navigation boxes
        "table.infobox",  # Sidebars (infoboxes)
        "table.metadata",  # Stub/disambiguation notices
        "div.thumb",  # Image captions
        "div.noprint",  # Print-hidden elements
        "style",
        "script",
    ]
    for sel in unwanted_selectors:
        for element in content.select(sel):
            element.decompose()

In [ ]:
def scrape_article(url: str) -> dict | None:
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        parsed = BeautifulSoup(response.text, "html.parser")

        title_tag = parsed.find("h1", {"id": "firstHeading"})
        title = title_tag.get_text().strip() if title_tag else ""

        content = parsed.select_one("#mw-content-text > div.mw-parser-output")
        if not content:
            content = parsed.select_one("#bodyContent div.mw-parser-output")

        if not content:
            print(f"No content found for {url}")
            return None

        clean_content(content)

        paragraphs = []
        for p in content.find_all("p"):
            txt = p.get_text().strip()
            if len(txt) > 30:
                txt = re.sub(r"\s+", " ", txt)
                paragraphs.append(txt)

        if not paragraphs:
            print(f"No valid paragraphs found for {url}")
            return None
        
        summary = paragraphs[0]
        full_text = "\n\n".join(paragraphs)

        categories = []
        cat_links = parsed.select("#mw-normal-catlinks ul li a")
        for cat_link in cat_links:
            categories.append(cat_link.get_text().strip())

        return {
            "title": title,
            "url": url,
            "summary": summary,
            "text": full_text,
            "categories": categories,
            "num_characters": len(full_text),
            "num_paragraphs": len(paragraphs),
        }

        
    except AttributeError as e:
        print(f"Error scraping {url}: {e}")
        return None

In [13]:
def main():
    scraped_urls = set()
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    scraped_urls.add(data["url"])
                except json.JSONDecodeError:
                    continue
        print(
            f"Found {len(scraped_urls)} existing records in {OUTPUT_FILE}. Continuing the scraping."
        )

    if len(scraped_urls) >= TARGET_ARTICLE_COUNT:
        print(f"Already scraped {len(scraped_urls)} articles. Exiting.")
        return

    print("STEP 1: Getting links to scrape")
    links_to_scrape = get_links_to_scrape()
    

    if len(links_to_scrape) < TARGET_ARTICLE_COUNT:
        print(f"Found only {len(links_to_scrape)} unique links.")

        if os.path.exists(ADDITIONAL_LINKS_FILE):
            with open(ADDITIONAL_LINKS_FILE, "r") as f:
                additional_links = [line.strip() for line in f if line.strip()]
                links_to_scrape.extend(additional_links)
                links_to_scrape = list(set(links_to_scrape))
                print(f"Loaded {len(additional_links)} additional links from {ADDITIONAL_LINKS_FILE}.")

        else:
            print("Extracting additional links to books from authors.")
            links_to_scrape = expand_list_of_links(links_to_scrape, links_to_authors_lm + links_to_authors_n + links_to_authors_p)
        
    
    print(f"Found {len(links_to_scrape)} unique links to scrape. Continuing")

    print(f"STEP 2: Scraping articles [{len(scraped_urls)}/{TARGET_ARTICLE_COUNT}]")
    with open(OUTPUT_FILE, "a", encoding="utf-8") as out_file:
        progress_bar = tqdm(
            total=TARGET_ARTICLE_COUNT, initial=len(scraped_urls), desc="Scraping"
        )

        for link in links_to_scrape:
            if len(scraped_urls) >= TARGET_ARTICLE_COUNT:
                print(f"Reached target of {TARGET_ARTICLE_COUNT} articles. Stopping.")
                break

            if link in scraped_urls:
                continue

            if len(scraped_urls) % 50 == 0:
                print(f"[{len(scraped_urls)}/{TARGET_ARTICLE_COUNT}]")

            article_data = scrape_article(link)

            if article_data and article_data["num_characters"] >= 300:
                out_file.write(json.dumps(article_data, ensure_ascii=False) + "\n")
                out_file.flush()
                scraped_urls.add(link)
                progress_bar.update(1)

            sleep(random.random() * REQUEST_DELAY_MULTIPLIER)
        progress_bar.close()

    print(f"Scraping completed. Successfully saved {len(scraped_urls)} articles to '{OUTPUT_FILE}'.")

In [14]:
main()

Found 1000 existing records in scraped_data.jsonl. Continuing the scraping.
STEP 1: Getting links to scrape
Found only 604 unique links.
Loaded 764 additional links from additional_links.txt.
Found 1085 unique links to scrape. Continuing
STEP 2: Scraping articles [1000/1200]


Scraping:  83%|████████▎ | 1000/1200 [00:00<?, ?it/s]

[1000/1200]


Scraping:  84%|████████▍ | 1013/1200 [00:23<05:33,  1.78s/it]

Error scraping https://en.wikipedia.org/wiki/Ifâ€”: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/If%C3%A2%E2%82%AC%E2%80%9D


...

Error scraping https://en.wikipedia.org/wiki/The_Confusions_of_Young_TĂ¶rless: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/The_Confusions_of_Young_T%C4%82%C2%B6rless


Scraping:  90%|████████▉ | 1075/1200 [02:50<06:46,  3.25s/it]

Error scraping https://en.wikipedia.org/wiki/The_Boat_Rocker?action=edit&redlink=1: 404 Client Error: Not Found for url: https://en.wikipedia.org/wiki/The_Boat_Rocker


Scraping:  90%|████████▉ | 1079/1200 [03:01<04:38,  2.30s/it]

Scraping completed. Successfully saved 1079 articles to 'scraped_data.jsonl'.
